In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Wealth_of_Healh_and_Nutrition

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Wealth_of_Healh_and_Nutrition


In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "src"))
from target import build_metabolic_targets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score

from sklearn.inspection import permutation_importance
%matplotlib inline

In [3]:
RF_FEATURES = [
    "energy_kcal", "protein_g", "carb_g", "sugar_g", "fat_total_g",
    "fat_sat_g", "fiber_g", "sodium_mg", "potassium_mg",
    "moderate_rec_min_week", "sedentary_min",
]
RF_OPTIONAL_FEATURES = [
    "sleep_hours", "vigorous_rec_min_week", "smoked_100_cigs",
    "avg_drinks_per_day",
]
FEATURES = RF_FEATURES + RF_OPTIONAL_FEATURES


In [4]:
TEST_UIDS_PATH = Path("splits/test_uids.csv")
TRAIN_UIDS_PATH = Path("splits/train_uids.csv")
DATA_PATH = Path("data/processed/nhanes_clean.csv")

df = pd.read_csv(DATA_PATH)
test_uid = pd.read_csv(TEST_UIDS_PATH)
train_uid = pd.read_csv(TRAIN_UIDS_PATH)
test_split = df[df["uid"].isin(test_uid["uid"])]
train_split = df[df["uid"].isin(train_uid["uid"])]
x_train = train_split[FEATURES]
y_train = train_split["metabolic_syndrome"]
x_test = test_split[FEATURES]
y_test = test_split["metabolic_syndrome"]

In [6]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=90 ,
    min_samples_leaf=30,
    max_features="log2",

)
rf.fit(x_train, y_train)

# Performance
y_train_pred = rf.predict(x_train)
y_test_pred = rf.predict(x_test)
print("Training Performance")
print(classification_report(y_train, y_train_pred))
print("Testing Performance")
print(classification_report(y_test, y_test_pred))

y_prob_train = rf.predict_proba(x_train)[:,1]
y_prob_test = rf.predict_proba(x_test)[:,1]
print(
    "Train AUC:",
    roc_auc_score(y_train, y_prob_train)
)
print(
    "Test AUC:",
    roc_auc_score(y_test, y_prob_test)
)

print("Confusion Matrix:")
print(confusion_matrix(
    y_test,
    y_test_pred,
))

Training Performance
              precision    recall  f1-score   support

         0.0       0.69      0.98      0.81      6015
         1.0       0.83      0.16      0.26      3129

    accuracy                           0.70      9144
   macro avg       0.76      0.57      0.54      9144
weighted avg       0.74      0.70      0.62      9144

Testing Performance
              precision    recall  f1-score   support

         0.0       0.67      0.96      0.79      1504
         1.0       0.55      0.09      0.15       783

    accuracy                           0.66      2287
   macro avg       0.61      0.52      0.47      2287
weighted avg       0.63      0.66      0.57      2287

Train AUC: 0.778936646877533
Test AUC: 0.6546272519768483
Confusion Matrix:
[[1449   55]
 [ 716   67]]


In [8]:
perm = permutation_importance(
    rf,
    x_test,
    y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=42,
)

importance = pd.DataFrame({
    "Feature": x_test.columns,
    "Importance": perm.importances_mean,
    "std": perm.importances_std
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

importance = importance[
    ~importance["Feature"].isin(["age", "sex"])
]

importance.head(5)


,Feature,Importance,std
12,vigorous_rec_min_week,0.078974,0.009668
14,avg_drinks_per_day,0.034599,0.005376
9,moderate_rec_min_week,0.005381,0.003423
13,smoked_100_cigs,0.004630,0.001767
0,energy_kcal,0.004228,0.002550
